# Pipeline Volume 3D — Approche Voxel

**Prérequis :** exécuter `depth_field_V3_2d.ipynb` jusqu'à la fin pour générer :

`data/processed/depth_field_v3_bundle.npz`

---

## Méthode géométrique vs méthode voxel

| Aspect | Géométrique (mesh) | **Voxel (ce notebook)** |
|---|---|---|
| Prérequis | Mesh **watertight** (étanche) | Nuage de points uniquement |
| Fragilité | Trous → volume faux ou impossible | Robuste aux nuages incomplets |
| Précision | Exacte si mesh parfait | Dépend de la taille du voxel |
| Remplissage intérieur | Implicite dans la géométrie | `scipy.ndimage.binary_fill_holes` sur grille 3D |
| Complexité | Reconstruction de surface (Poisson, BPA…) | Projection en grille → opération binaire |

### Principe de la voxelisation remplie

1. Créer une `VoxelGrid` Open3D depuis le nuage de points → **coquille de surface** uniquement
2. Extraire les indices de grille `(i, j, k)` des voxels occupés
3. Projeter dans un tableau numpy 3D binaire
4. Appliquer `binary_fill_holes` (axe par axe ou 3D) pour remplir l'intérieur
5. Compter les voxels remplis → **volume = N_remplis × voxel_size³**

In [21]:
# ── Cellule 1 : Chargement du bundle ──────────────────────────────────────────
from pathlib import Path
import numpy as np
from PIL import Image
import open3d as o3d
import cv2

EXPORT_PATH = Path("C:/Users/mvm/open3d_vision/data/processed") / "depth_field_v3_bundle.npz"
if not EXPORT_PATH.is_file():
    raise FileNotFoundError(
        "Fichier introuvable : exécuter d'abord depth_field_V3_2d.ipynb (cellule d'export).\n"
        f"Attendu : {EXPORT_PATH}"
    )

z = np.load(EXPORT_PATH)
depth_1       = z["depth_1"].astype(np.float64)
depth_2_aligned = z["depth_2_remapped"].astype(np.float64)
mask_combined = z["mask_combined"].astype(np.uint8)
rgb1          = z["rgb1"].astype(np.uint8)
rgb2          = z["rgb2"].astype(np.uint8)
pad           = int(z["pad"])

image_cropped_1 = Image.fromarray(rgb1)
image_cropped_2 = Image.fromarray(rgb2)

h_d1, w_d1 = depth_1.shape
depth_2_aligned = np.asarray(depth_2_aligned, dtype=np.float64)
if depth_2_aligned.shape != (h_d1, w_d1):
    depth_2_aligned = cv2.resize(
        depth_2_aligned.astype(np.float32), (w_d1, h_d1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)

depth_1_masked = np.where(mask_combined == 255, depth_1,          np.nan)
depth_2_masked = np.where(mask_combined == 255, depth_2_aligned,  np.nan)

print(f"Chargé : {EXPORT_PATH}")
print(f"Profondeur : {h_d1}×{w_d1}  |  pad={pad}")
print(f"Pixels valides masque : {int((mask_combined == 255).sum())}")

Chargé : C:\Users\mvm\open3d_vision\data\processed\depth_field_v3_bundle.npz
Profondeur : 1811×1350  |  pad=16
Pixels valides masque : 969615


In [22]:
# ── Cellule 2 : Construction des nuages 3D + normalisation métrique ───────────
#
# Convention :
#   X, Y  = position pixel normalisée par max(h, w)
#   Z     = valeur depth_masked (proche caméra = valeur haute)
#   Échelle métrique : la plus grande dimension XY = 1 m
#
# NOTE sur l'axe Z : les depth maps MiDaS/DPT produisent des valeurs
# de disparité relative (haut = proche). On conserve cet axe tel quel
# pour la voxelisation ; le sens n'affecte pas le calcul de volume.

def build_pcd_from_masked_depth(depth_masked, rgb_uint8, tint=None, tint_mix=0.0):
    """Construit un PointCloud Open3D depuis une depth map masquée (NaN = fond)."""
    h, w = depth_masked.shape
    s_xy = float(max(h, w))
    xx, yy = np.meshgrid(np.arange(w), np.arange(h), indexing="xy")

    x_n = xx.astype(np.float64) / s_xy
    y_n = yy.astype(np.float64) / s_xy
    z_n = np.where(np.isfinite(depth_masked), depth_masked.astype(np.float64), np.nan)

    pts  = np.stack([x_n, y_n, z_n], axis=-1).reshape(-1, 3)
    valid = np.isfinite(pts[:, 2])

    rgb_arr = np.asarray(rgb_uint8, dtype=np.uint8)
    if rgb_arr.shape[0] != h or rgb_arr.shape[1] != w:
        rgb_arr = np.asarray(
            Image.fromarray(rgb_arr).resize((w, h), Image.Resampling.LANCZOS),
            dtype=np.uint8,
        )
    cols = rgb_arr.reshape(-1, 3).astype(np.float64) / 255.0
    if tint is not None:
        cols = np.clip((1 - tint_mix) * cols + tint_mix * np.asarray(tint), 0.0, 1.0)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts[valid])
    pcd.colors = o3d.utility.Vector3dVector(cols[valid])
    return pcd


def scale_pcd_to_metric(pcd):
    """Normalise un nuage : la plus grande dimension = 1 m.
    Retourne (pcd_scaled, scale_factor) où scale_factor est le facteur appliqué."""
    pts = np.asarray(pcd.points)
    extent = pts.max(axis=0) - pts.min(axis=0)
    max_ext = float(extent.max())
    if max_ext < 1e-12:
        return pcd, 1.0
    scale = 1.0 / max_ext
    pcd_s = o3d.geometry.PointCloud(pcd)
    pcd_s.points = o3d.utility.Vector3dVector(pts * scale)
    return pcd_s, scale


# Nuages bruts (unités normalisées par max(h,w))
pcd_1_raw = build_pcd_from_masked_depth(depth_1_masked, rgb1,  tint=[0.92,0.38,0.28], tint_mix=0.45)
pcd_2_raw = build_pcd_from_masked_depth(depth_2_masked, rgb2,  tint=[0.25,0.55,0.95], tint_mix=0.45)

# Nuage de différence (Z = depth_1 - depth_2)
diff_masked = np.where(
    np.isfinite(depth_1_masked) & np.isfinite(depth_2_masked),
    depth_1_masked - depth_2_masked,
    np.nan,
)
rgb_diff_display = np.asarray(image_cropped_1, dtype=np.uint8)
pcd_diff_raw = build_pcd_from_masked_depth(diff_masked, rgb_diff_display, tint=[0.2,0.8,0.4], tint_mix=0.5)

# Mise à l'échelle métrique (1 m = max dimension)
pcd_1, scale_1 = scale_pcd_to_metric(pcd_1_raw)
pcd_2, scale_2 = scale_pcd_to_metric(pcd_2_raw)
pcd_diff, scale_diff = scale_pcd_to_metric(pcd_diff_raw)

pts1 = np.asarray(pcd_1.points)
pts2 = np.asarray(pcd_2.points)
ptsd = np.asarray(pcd_diff.points)

print("=== Nuages de points ===")
for name, pts, sc in [("Image 1", pts1, scale_1), ("Image 2", pts2, scale_2), ("Différence", ptsd, scale_diff)]:
    ext = pts.max(axis=0) - pts.min(axis=0)
    print(f"  {name:12s}: {len(pts):>7} pts | facteur échelle={sc:.4g} | "
          f"extent XYZ=[{ext[0]:.3f}, {ext[1]:.3f}, {ext[2]:.3f}] m")

=== Nuages de points ===
  Image 1     :  969615 pts | facteur échelle=1.466 | extent XYZ=[0.957, 1.000, 0.915] m
  Image 2     :  969615 pts | facteur échelle=1.466 | extent XYZ=[0.957, 1.000, 0.776] m
  Différence  :  969615 pts | facteur échelle=1.466 | extent XYZ=[0.957, 1.000, 0.738] m


In [23]:
# ── Cellule 3 : Voxelisation de surface (Open3D) ──────────────────────────────
#
# La VoxelGrid Open3D ne couvre que les voxels touchés par au moins un point
# du nuage : c'est une *coquille de surface*, pas un solide.
# Ce volume est donc une SOUS-ESTIMATION du volume réel de l'objet.

VOXEL_SIZE_VIZ = 0.01  # 1 cm pour la visualisation

vg_1 = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_1, voxel_size=VOXEL_SIZE_VIZ)
vg_2 = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_2, voxel_size=VOXEL_SIZE_VIZ)

n_surf_1 = len(vg_1.get_voxels())
n_surf_2 = len(vg_2.get_voxels())
vol_surf_1 = n_surf_1 * VOXEL_SIZE_VIZ ** 3
vol_surf_2 = n_surf_2 * VOXEL_SIZE_VIZ ** 3

print(f"Voxel size : {VOXEL_SIZE_VIZ*100:.1f} cm")
print(f"Image 1 — voxels surface : {n_surf_1:>6}  →  volume surface : {vol_surf_1:.6f} m³")
print(f"Image 2 — voxels surface : {n_surf_2:>6}  →  volume surface : {vol_surf_2:.6f} m³")
print("(Sous-estimation : intérieur de l'objet non compté)")

# Visualisation : VoxelGrid image 1 + nuage image 2 pour comparaison
pcd_2_vis = o3d.geometry.PointCloud(pcd_2)
pcd_2_vis.paint_uniform_color([0.25, 0.55, 0.95])
o3d.visualization.draw_geometries(
    [vg_1, pcd_2_vis],
    window_name=f"VoxelGrid surface — Image 1 (rouge) + nuage Image 2 (bleu) | voxel={VOXEL_SIZE_VIZ*100:.0f} cm",
)

Voxel size : 1.0 cm
Image 1 — voxels surface :  19253  →  volume surface : 0.019253 m³
Image 2 — voxels surface :  15707  →  volume surface : 0.015707 m³
(Sous-estimation : intérieur de l'objet non compté)


In [39]:
# ── Cellule 4 : Remplissage intérieur 3D (scipy) ──────────────────────────────
#
# Algorithme :
#   1. Extraire les indices (i,j,k) des voxels de surface depuis la VoxelGrid
#   2. Les placer dans un tableau numpy bool 3D avec marge de 1 voxel
#   3. binary_fill_holes sur chaque coupe (axe Z, puis XY) pour remplir l'intérieur
#   4. Volume = N_voxels_remplis × voxel_size³
#
# Remarque : binary_fill_holes fonctionne coupe par coupe (2D) ou en 3D.
# La version 3D est plus rigoureuse mais peut laisser des trous ouverts
# si la coquille n'est pas fermée. On utilise les deux et on garde le plus grand.

from scipy.ndimage import binary_fill_holes


def voxelgrid_to_binary_array(vg):
    """Extrait les indices (i,j,k) d'une VoxelGrid Open3D et retourne
    (grid_bool, grid_min_indices) où grid_bool est le tableau 3D binaire
    avec marge de 1 voxel de chaque côté."""
    voxels = vg.get_voxels()
    if not voxels:
        return np.zeros((3, 3, 3), dtype=bool), np.zeros(3, dtype=np.int64)
    indices = np.array([v.grid_index for v in voxels], dtype=np.int64)
    g_min = indices.min(axis=0)
    g_max = indices.max(axis=0)
    shape  = tuple((g_max - g_min + 3).tolist())  # marge 1 voxel chaque côté
    grid   = np.zeros(shape, dtype=bool)
    shifted = indices - g_min + 1  # décalage pour la marge
    grid[shifted[:, 0], shifted[:, 1], shifted[:, 2]] = True
    return grid, g_min


def fill_volume_3d(grid_surface, axis_fill="all"):
    """Remplit l'intérieur d'une coquille binaire 3D.

    axis_fill:
      'z'   : remplissage coupe par coupe selon Z (robuste si coquille ouverte en Z)
      'y'   : remplissage coupe par coupe selon Y
      'x'   : remplissage coupe par coupe selon X
      'all' : remplissage 3D global puis union des trois axes
    """
    # Remplissage 3D global
    filled_3d = binary_fill_holes(grid_surface)

    if axis_fill == "all":
        # Remplissage coupe par coupe sur les trois axes (plus robuste)
        nx, ny, nz = grid_surface.shape
        filled_x = np.zeros_like(grid_surface)
        filled_y = np.zeros_like(grid_surface)
        filled_z = np.zeros_like(grid_surface)
        for i in range(nx):
            filled_x[i] = binary_fill_holes(grid_surface[i])
        for j in range(ny):
            filled_y[:, j, :] = binary_fill_holes(grid_surface[:, j, :])
        for k in range(nz):
            filled_z[:, :, k] = binary_fill_holes(grid_surface[:, :, k])
        # Union des quatre versions (conservatrice)
        filled = filled_3d | filled_x | filled_y | filled_z
    else:
        filled = filled_3d

    return filled


VOXEL_SIZE = 0.01  # 1 cm

# --- Image 1 ---
vg_1 = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_1, voxel_size=VOXEL_SIZE)
grid_1, gmin_1 = voxelgrid_to_binary_array(vg_1)
grid_1_filled  = fill_volume_3d(grid_1, axis_fill="all")

n_surf_1   = int(grid_1.sum())
n_fill_1   = int(grid_1_filled.sum())
vol_surf_1 = n_surf_1 * VOXEL_SIZE ** 3
vol_fill_1 = n_fill_1 * VOXEL_SIZE ** 3

# --- Image 2 ---
vg_2 = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_2, voxel_size=VOXEL_SIZE)
grid_2, gmin_2 = voxelgrid_to_binary_array(vg_2)
grid_2_filled  = fill_volume_3d(grid_2, axis_fill="all")

n_surf_2   = int(grid_2.sum())
n_fill_2   = int(grid_2_filled.sum())
vol_surf_2 = n_surf_2 * VOXEL_SIZE ** 3
vol_fill_2 = n_fill_2 * VOXEL_SIZE ** 3

print(f"Voxel size : {VOXEL_SIZE*100:.1f} cm")
print()
print("             │  Voxels surface │  Voxels remplis │  Vol. surface (m³)  │  Vol. rempli (m³)")
print("─────────────┼─────────────────┼─────────────────┼─────────────────────┼──────────────────")
print(f" Image 1     │  {n_surf_1:>13,}  │  {n_fill_1:>13,}  │  {vol_surf_1:>17.6f}  │  {vol_fill_1:.6f}")
print(f" Image 2     │  {n_surf_2:>13,}  │  {n_fill_2:>13,}  │  {vol_surf_2:>17.6f}  │  {vol_fill_2:.6f}")
print()
print(f" Ratio remplissage/surface — Image 1 : {n_fill_1/max(n_surf_1,1):.2f}x")
print(f" Ratio remplissage/surface — Image 2 : {n_fill_2/max(n_surf_2,1):.2f}x")

Voxel size : 1.0 cm

             │  Voxels surface │  Voxels remplis │  Vol. surface (m³)  │  Vol. rempli (m³)
─────────────┼─────────────────┼─────────────────┼─────────────────────┼──────────────────
 Image 1     │         19,253  │         67,059  │           0.019253  │  0.067059
 Image 2     │         15,707  │         24,117  │           0.015707  │  0.024117

 Ratio remplissage/surface — Image 1 : 3.48x
 Ratio remplissage/surface — Image 2 : 1.54x


In [40]:
# ── Cellule 5 : Analyse multi-résolution ──────────────────────────────────────
#
# Teste plusieurs tailles de voxel pour observer la convergence du volume.
# Un volume stable à différentes résolutions indique une estimation fiable.

import matplotlib
matplotlib.use("Agg")  # backend non-interactif pour Jupyter
import matplotlib.pyplot as plt

VOXEL_SIZES = [0.005, 0.008, 0.01, 0.015, 0.02, 0.03, 0.05]

results = []
for vs in VOXEL_SIZES:
    # Image 1
    vg  = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_1, voxel_size=vs)
    g, _= voxelgrid_to_binary_array(vg)
    gf  = fill_volume_3d(g, axis_fill="all")
    ns1 = int(g.sum())
    nf1 = int(gf.sum())

    # Image 2
    vg2  = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_2, voxel_size=vs)
    g2, _= voxelgrid_to_binary_array(vg2)
    gf2  = fill_volume_3d(g2, axis_fill="all")
    ns2  = int(g2.sum())
    nf2  = int(gf2.sum())

    results.append({
        "vs_cm"    : vs * 100,
        "ns1": ns1, "nf1": nf1, "vs1": ns1 * vs**3, "vf1": nf1 * vs**3,
        "ns2": ns2, "nf2": nf2, "vs2": ns2 * vs**3, "vf2": nf2 * vs**3,
    })
    print(f"  vs={vs*100:5.1f} cm | "
          f"Img1: surf={ns1:>6} ({ns1*vs**3:.5f} m³)  fill={nf1:>6} ({nf1*vs**3:.5f} m³) | "
          f"Img2: surf={ns2:>6} ({ns2*vs**3:.5f} m³)  fill={nf2:>6} ({nf2*vs**3:.5f} m³)")

# Courbe de convergence
vs_cm  = [r["vs_cm"] for r in results]
vf1_l  = [r["vf1"]   for r in results]
vf2_l  = [r["vf2"]   for r in results]
vs1_l  = [r["vs1"]   for r in results]
vs2_l  = [r["vs2"]   for r in results]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, name, vf_l, vs_l, color_f, color_s in [
    (axes[0], "Image 1", vf1_l, vs1_l, "#e06040", "#f0a080"),
    (axes[1], "Image 2", vf2_l, vs2_l, "#3060d0", "#80a0f0"),
]:
    ax.plot(vs_cm, vf_l, "o-", color=color_f, lw=2, label="Volume rempli")
    ax.plot(vs_cm, vs_l, "s--", color=color_s, lw=1.5, label="Volume surface")
    ax.set_xlabel("Taille voxel (cm)")
    ax.set_ylabel("Volume (m³)")
    ax.set_title(f"{name} — Convergence volume voxel")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(left=0)

plt.tight_layout()
plt.savefig("C:/Users/mvm/open3d_vision/data/processed/volume_convergence.png", dpi=120)
plt.show()
print("Courbe sauvegardée : data/processed/volume_convergence.png")

  vs=  0.5 cm | Img1: surf= 72613 (0.00908 m³)  fill=245781 (0.03072 m³) | Img2: surf= 59462 (0.00743 m³)  fill=138780 (0.01735 m³)
  vs=  0.8 cm | Img1: surf= 29808 (0.01526 m³)  fill=110016 (0.05633 m³) | Img2: surf= 24186 (0.01238 m³)  fill= 42510 (0.02177 m³)
  vs=  1.0 cm | Img1: surf= 19253 (0.01925 m³)  fill= 67059 (0.06706 m³) | Img2: surf= 15707 (0.01571 m³)  fill= 24117 (0.02412 m³)
  vs=  1.5 cm | Img1: surf=  8628 (0.02912 m³)  fill= 22902 (0.07729 m³) | Img2: surf=  7064 (0.02384 m³)  fill=  9708 (0.03276 m³)
  vs=  2.0 cm | Img1: surf=  4888 (0.03910 m³)  fill= 10886 (0.08709 m³) | Img2: surf=  4013 (0.03210 m³)  fill=  4951 (0.03961 m³)
  vs=  3.0 cm | Img1: surf=  2189 (0.05910 m³)  fill=  3608 (0.09742 m³) | Img2: surf=  1841 (0.04971 m³)  fill=  2223 (0.06002 m³)
  vs=  5.0 cm | Img1: surf=   795 (0.09938 m³)  fill=  1078 (0.13475 m³) | Img2: surf=   683 (0.08538 m³)  fill=   745 (0.09313 m³)
Courbe sauvegardée : data/processed/volume_convergence.png


C:\Users\mvm\AppData\Local\Temp\ipykernel_30632\616907568.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [41]:
# ── Cellule 6 : Volume différentiel (Image 1 − Image 2) ───────────────────────
#
# Le nuage de différence (Z = depth_1 − depth_2) représente la variation
# de profondeur entre les deux acquisitions.
# On sépare :
#   - les voxels positifs  (Z > 0 → Image 1 plus proche caméra → "plus de matière")
#   - les voxels négatifs  (Z < 0 → Image 2 plus proche caméra)
# ce qui permet d'estimer un volume de différence signé.
#
# On applique aussi la voxelisation remplie sur le nuage différence complet
# (valeur absolue) pour obtenir le volume total affecté.

VOXEL_SIZE_DIFF = 0.01

# --- Nuage différence complet (valeurs positives et négatives) ---
ptsd = np.asarray(pcd_diff.points)
z_diff = ptsd[:, 2]

z_pos_mask = z_diff >  1e-9
z_neg_mask = z_diff < -1e-9

print(f"Pixels diff > 0 (img1 plus proche) : {z_pos_mask.sum():>7}")
print(f"Pixels diff < 0 (img2 plus proche) : {z_neg_mask.sum():>7}")
print(f"Pixels diff ≈ 0                    : {(~z_pos_mask & ~z_neg_mask).sum():>7}")
print()


def pcd_from_pts_cols(pts, cols):
    p = o3d.geometry.PointCloud()
    p.points = o3d.utility.Vector3dVector(pts)
    p.colors = o3d.utility.Vector3dVector(cols)
    return p


cols_diff = np.asarray(pcd_diff.colors)

# Sous-nuages positif / négatif
pcd_pos = pcd_from_pts_cols(ptsd[z_pos_mask], cols_diff[z_pos_mask])
pcd_neg = pcd_from_pts_cols(ptsd[z_neg_mask], cols_diff[z_neg_mask])

# Voxelisation remplie sur le nuage complet (Z absolu pour obtenir les voxels)
# On utilise |Z| pour que les régions positives et négatives occupent le même espace voxel
pts_abs = ptsd.copy()
pts_abs[:, 2] = np.abs(ptsd[:, 2])
pcd_diff_abs = pcd_from_pts_cols(pts_abs, cols_diff)

vg_diff = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_diff_abs, voxel_size=VOXEL_SIZE_DIFF)
g_diff, _ = voxelgrid_to_binary_array(vg_diff)
gf_diff   = fill_volume_3d(g_diff, axis_fill="all")

# Volumes des sous-nuages signés
results_diff = {}
for label, pcdi, col in [("positif", pcd_pos, [0.9,0.4,0.2]), ("négatif", pcd_neg, [0.2,0.5,0.9])]:
    if len(np.asarray(pcdi.points)) < 4:
        results_diff[label] = {"ns": 0, "nf": 0, "vs": 0.0, "vf": 0.0}
        print(f"Diff {label}: nuage vide — ignoré.")
        continue
    vgi  = o3d.geometry.VoxelGrid.create_from_point_cloud(pcdi, voxel_size=VOXEL_SIZE_DIFF)
    gi, _= voxelgrid_to_binary_array(vgi)
    gfi  = fill_volume_3d(gi, axis_fill="all")
    ns   = int(gi.sum())
    nf   = int(gfi.sum())
    vs_m3 = ns * VOXEL_SIZE_DIFF**3
    vf_m3 = nf * VOXEL_SIZE_DIFF**3
    results_diff[label] = {"ns": ns, "nf": nf, "vs": vs_m3, "vf": vf_m3}
    print(f"Diff {label:8s} : surface={ns:>5} vox ({vs_m3:.6f} m³) | rempli={nf:>5} vox ({vf_m3:.6f} m³)")

nd = int(g_diff.sum())
ndf = int(gf_diff.sum())
print(f"\nDiff total (|Z|): surface={nd:>5} vox ({nd*VOXEL_SIZE_DIFF**3:.6f} m³) | "
      f"rempli={ndf:>5} vox ({ndf*VOXEL_SIZE_DIFF**3:.6f} m³)")

# Visualisation nuages positif (rouge) et négatif (bleu)
pcd_pos_v = o3d.geometry.PointCloud(pcd_pos); pcd_pos_v.paint_uniform_color([0.9, 0.35, 0.2])
pcd_neg_v = o3d.geometry.PointCloud(pcd_neg); pcd_neg_v.paint_uniform_color([0.2, 0.45, 0.9])
o3d.visualization.draw_geometries(
    [pcd_pos_v, pcd_neg_v],
    window_name="Différence Z : rouge=Img1>Img2 | bleu=Img2>Img1",
)

Pixels diff > 0 (img1 plus proche) :   30047
Pixels diff < 0 (img2 plus proche) :  939568
Pixels diff ≈ 0                    :       0

Diff positif  : surface= 1117 vox (0.001117 m³) | rempli= 1976 vox (0.001976 m³)
Diff négatif  : surface=17361 vox (0.017361 m³) | rempli=26042 vox (0.026042 m³)

Diff total (|Z|): surface=18055 vox (0.018055 m³) | rempli=24716 vox (0.024716 m³)


In [42]:
# ── Cellule 7 : Visualisation 3D des voxels de remplissage ───────────────────
#
# Objectif visuel :
#   - voxels de SURFACE conservés dans la couleur de l'image
#   - voxels AJOUTÉS par le remplissage (grid_filled & ~grid_surface) en VERT

VOXEL_SIZE_VIZ2 = 0.01  # doit correspondre à VOXEL_SIZE de la cellule 4


def binary_grid_to_pointcloud(grid_bool, vg_origin, voxel_size, color=None):
    """Convertit une grille bool en points au centre des voxels Open3D."""
    idx = np.argwhere(grid_bool).astype(np.float64)
    if idx.size == 0:
        pcd = o3d.geometry.PointCloud()
        return pcd

    # Retire la marge (+1) introduite dans voxelgrid_to_binary_array
    idx -= 1.0
    pts = idx * voxel_size + np.asarray(vg_origin, dtype=np.float64)
    pts += 0.5 * voxel_size  # centre voxel

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    if color is not None:
        pcd.paint_uniform_color(color)
    return pcd


def pointcloud_to_voxelgrid(pcd, voxel_size):
    if len(np.asarray(pcd.points)) == 0:
        return o3d.geometry.VoxelGrid()
    return o3d.geometry.VoxelGrid.create_from_point_cloud(pcd, voxel_size=voxel_size)


# Origines des VoxelGrids de surface (cellule 4)
origin_1 = np.asarray(vg_1.origin, dtype=np.float64)
origin_2 = np.asarray(vg_2.origin, dtype=np.float64)

# Masques : surface vs voxels créés par remplissage
grid_1_added = grid_1_filled & (~grid_1)
grid_2_added = grid_2_filled & (~grid_2)

# PointClouds puis VoxelGrids
pcd_1_surface = binary_grid_to_pointcloud(grid_1, origin_1, VOXEL_SIZE_VIZ2, color=[0.92, 0.38, 0.28])
pcd_2_surface = binary_grid_to_pointcloud(grid_2, origin_2, VOXEL_SIZE_VIZ2, color=[0.25, 0.55, 0.95])

# Tous les voxels de remplissage en vert
pcd_1_added = binary_grid_to_pointcloud(grid_1_added, origin_1, VOXEL_SIZE_VIZ2, color=[0.10, 0.85, 0.20])
pcd_2_added = binary_grid_to_pointcloud(grid_2_added, origin_2, VOXEL_SIZE_VIZ2, color=[0.10, 0.85, 0.20])

# Décalage visuel de l'image 2
offset = np.array([1.3, 0.0, 0.0], dtype=np.float64)
if len(np.asarray(pcd_2_surface.points)):
    pcd_2_surface.points = o3d.utility.Vector3dVector(np.asarray(pcd_2_surface.points) + offset)
if len(np.asarray(pcd_2_added.points)):
    pcd_2_added.points = o3d.utility.Vector3dVector(np.asarray(pcd_2_added.points) + offset)

vg_1_surface = pointcloud_to_voxelgrid(pcd_1_surface, VOXEL_SIZE_VIZ2)
vg_2_surface = pointcloud_to_voxelgrid(pcd_2_surface, VOXEL_SIZE_VIZ2)
vg_1_added   = pointcloud_to_voxelgrid(pcd_1_added, VOXEL_SIZE_VIZ2)
vg_2_added   = pointcloud_to_voxelgrid(pcd_2_added, VOXEL_SIZE_VIZ2)

n1_added = int(grid_1_added.sum())
n2_added = int(grid_2_added.sum())
print(f"Image 1 — voxels ajoutés (remplissage) : {n1_added} → {n1_added * VOXEL_SIZE_VIZ2**3:.6f} m³")
print(f"Image 2 — voxels ajoutés (remplissage) : {n2_added} → {n2_added * VOXEL_SIZE_VIZ2**3:.6f} m³")
print("Couleurs : surface=rouge/bleu, remplissage=vert")

frame_1 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.08)
frame_2 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.08)
frame_2.translate(offset)

o3d.visualization.draw_geometries(
    [vg_1_surface, vg_1_added, vg_2_surface, vg_2_added, frame_1, frame_2],
    window_name=(
        "Voxelisation remplie — voxels ajoutés en vert | "
        f"Img1 verts: {n1_added} | Img2 verts: {n2_added}"
    ),
)

Image 1 — voxels ajoutés (remplissage) : 47806 → 0.047806 m³
Image 2 — voxels ajoutés (remplissage) : 8410 → 0.008410 m³
Couleurs : surface=rouge/bleu, remplissage=vert


In [43]:
# ── Cellule 8 : Tableau récapitulatif — toutes méthodes ───────────────────────
#
# Comparaison finale :
#   A) Voxel surface (sous-estimation)
#   B) Voxel + remplissage intérieur scipy (estimation principale)

from torch import NoneType


VOXEL_SIZE_VIZ = 0.01  # 1 cm pour la visualisation

vg_1 = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_1, voxel_size=VOXEL_SIZE_VIZ)
vg_2 = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_2, voxel_size=VOXEL_SIZE_VIZ)

vf1_surf = n_surf_1  # défini cellule 4
vf2_surf = n_surf_2
vol_a1   = vf1_surf * VOXEL_SIZE_FINAL**3
vol_a2   = vf2_surf * VOXEL_SIZE_FINAL**3

# Voxel rempli (cellule 4)
vol_b1   = n_fill_1 * VOXEL_SIZE_FINAL**3
vol_b2   = n_fill_2 * VOXEL_SIZE_FINAL**3

# Enveloppe convexe (via Open3D)
def convex_hull_volume(pcd):
    try:
        hull_mesh, _ = pcd.compute_convex_hull()
        if hull_mesh.is_watertight():
            return float(np.abs(hull_mesh.get_volume()))
    except Exception:
        pass
    return None

vol_c1 = convex_hull_volume(pcd_1)
vol_c2 = convex_hull_volume(pcd_2)

# Volume mesh géométrique (si disponible dans l'environnement — issu du notebook V3)
vol_d1 = None
vol_d2 = None
for varname, target_idx in [("mesh_super_1", 1), ("mesh_2", 2)]:
    if varname in globals():
        m = globals()[varname]
        if hasattr(m, "is_watertight") and m.is_watertight():
            try:
                v = float(np.abs(m.get_volume()))
                if target_idx == 1: vol_d1 = v
                else: vol_d2 = v
            except Exception:
                pass

# Récupère le volume différentiel (cellule 6)
vol_diff_fill = ndf * VOXEL_SIZE_DIFF**3  # grille diff totale remplie
vol_diff_pos  = results_diff.get("positif", {}).get("vf", None)
vol_diff_neg  = results_diff.get("négatif", {}).get("vf", None)

# ─── Affichage ────────────────────────────────────────────────────────────────
SEP = "═" * 80
print(SEP)
print(f"  RÉCAPITULATIF VOLUMES  (voxel_size = {VOXEL_SIZE_FINAL*100:.0f} cm)")
print(SEP)
print(f"  {'Méthode':<38} {'Image 1 (m³)':>14}  {'Image 2 (m³)':>14} {'Différence (m³)' :>14} {'Différence (%)' :>14}")
print("─" * 70)

def fmt(v):
    return f"{v:>14.6f}" if v is not None else f"{'N/A':>14}"

def fmt_percent(v1, v2):
    if v1 is not None and v2 is not None and max(abs(v1), abs(v2)) > 0:
        pct = abs(v1 - v2) / max(abs(v1), abs(v2)) * 100
        return f"{pct:>14.2f}"
    else:
        return f"{'N/A':>14}"

rows_summary = [
    ("A  Voxel surface seule (sous-estim.)",  vol_a1, vol_a2, np.abs(vol_a1-vol_a2)),
    ("B  Voxel + fill_holes scipy (★ estimat.)", vol_b1, vol_b2, np.abs(vol_b1-vol_b2)),
    ("C  Enveloppe convexe (sur-estim.)",     vol_c1, vol_c2, np.abs(vol_c1-vol_c2)),
    ("D  Mesh géométrique (si watertight)", vol_d1, vol_d2, 0 if vol_d1 is None or vol_d2 is None else np.abs(vol_d1 - vol_d2)),
]
for name, v1, v2, v3 in rows_summary:
    percent_diff = fmt_percent(v1, v2)
    print(f"  {name:<38} {fmt(v1)}  {fmt(v2)} {fmt(v3)}{percent_diff}")

print()
print("─" * 70)
print(f"  Volume DIFFÉRENTIEL (|Img1 − Img2|, voxel rempli)      {fmt(vol_diff_fill)}")
if vol_diff_pos is not None:
    print(f"    └ zone Img1 > Img2 (remplie)                         {fmt(vol_diff_pos)}")
if vol_diff_neg is not None:
    print(f"    └ zone Img2 > Img1 (remplie)                         {fmt(vol_diff_neg)}")
print(SEP)

# Ratios
if vol_b1 and vol_c1:
    print(f"\n  Ratio B/C Img1 (plénitude vs convexe) : {vol_b1/vol_c1:.3f}")
if vol_b2 and vol_c2:
    print(f"  Ratio B/C Img2 (plénitude vs convexe) : {vol_b2/vol_c2:.3f}")
if vol_b1 and vol_b2:
    print(f"\n  Différence relative Img1 vs Img2 (méthode B) : "
          f"{abs(vol_b1 - vol_b2) / max(vol_b1, vol_b2) * 100:.2f} %")

print()
print("  Légende :")
print("    A = voxels touchés par le nuage (coquille) → sous-estimation garantie")
print("    B = A + remplissage intérieur scipy (recommandé)")
print("    C = enveloppe convexe → borne haute (sur-estime si forme concave)")
print("    D = volume exact du mesh (uniquement si mesh watertight, sinon N/A)")

════════════════════════════════════════════════════════════════════════════════
  RÉCAPITULATIF VOLUMES  (voxel_size = 1 cm)
════════════════════════════════════════════════════════════════════════════════
  Méthode                                  Image 1 (m³)    Image 2 (m³) Différence (m³) Différence (%)
──────────────────────────────────────────────────────────────────────
  A  Voxel surface seule (sous-estim.)         0.019253        0.015707       0.003546         18.42
  B  Voxel + fill_holes scipy (★ estimat.)       0.067059        0.024117       0.042942         64.04
  C  Enveloppe convexe (sur-estim.)            0.207804        0.131270       0.076534         36.83
  D  Mesh géométrique (si watertight)               N/A             N/A       0.000000           N/A

──────────────────────────────────────────────────────────────────────
  Volume DIFFÉRENTIEL (|Img1 − Img2|, voxel rempli)            0.024716
    └ zone Img1 > Img2 (remplie)                               0.0019